# exp10 - clean-pipeline submission notebook

Fine-tuned UNet+transformer (dataset vitaliineibauer/biohub-ft-weights) -> predict + ILP on 2 GPUs -> gap closing + post-hoc divisions -> submission.csv. Optional self-check on the 12-video holdout with the official metric.


In [ ]:
import os, json
os.environ["BIOHUB_HOLDOUT_JSON"] = json.dumps({"holdout": ["44b6_1d530831", "44b6_74d0c52e", "44b6_7a302da0", "44b6_87bba6c4", "44b6_9be80b04", "44b6_e28840c6", "6bba_43fea39d", "6bba_74686d6a", "6bba_b204cac7", "6bba_cdcfe533", "6bba_f20478e9", "6bba_fbc898dc"]})
os.environ.setdefault("BIOHUB_WEIGHTS_SUB", "exp03B")
os.environ.setdefault("BIOHUB_SUBMIT_VALIDATE", "1")   # self-check on hold12 (set 0 for the real submission run)
os.environ.setdefault("BIOHUB_DET_THRESHOLD", "0.965")
os.environ.setdefault("BIOHUB_PP_GAP_UM", "5.0")
os.environ.setdefault("BIOHUB_PP_DIRECT_UM", "8.0")
os.environ.setdefault("BIOHUB_ILP_DIVISION_WEIGHT", "0.6")   # v2.1: ILP creates divisions itself
os.environ.setdefault("BIOHUB_PP_DIV_PARENT_UM", "0")      # v2.1: post-hoc divisions OFF
os.environ.setdefault("BIOHUB_PP_DIV_SISTER_UM", "12.0")
print({k: v for k, v in os.environ.items() if k.startswith("BIOHUB_") and k != "BIOHUB_HOLDOUT_JSON"})


In [ ]:
# --- Setup header: constants expected by the dependency/repo helpers (from baseline cell 8) ---
from __future__ import annotations
import csv, importlib.util, json, math, os, shutil, subprocess, tempfile, zipfile, sys, time
from pathlib import Path

COMPETITION = "biohub-cell-tracking-during-development"
COMP_DIR_CANDIDATES = [Path(f"/kaggle/input/competitions/{COMPETITION}"), Path(f"/kaggle/input/{COMPETITION}")]
COMP_DIR = next((p for p in COMP_DIR_CANDIDATES if p.exists()), COMP_DIR_CANDIDATES[0])
TRAIN_DIR = COMP_DIR / "train"
TEST_DIR = COMP_DIR / "test"
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
REPO_DIR = WORKING_DIR / "tracking_repo"
METHOD = "unet_transformer"
WEIGHTS_RELATIVE = f"weights/{METHOD}/split_0/edge_predictor_best.pth"
TARGET_ARTIFACT_SLUG = os.environ.get("BIOHUB_TARGET_ARTIFACT_SLUG", "biohub-tracking-support-pack-50ep-v1")
PRIMARY_ARTIFACT_MANIFEST = Path(os.environ.get(
    "BIOHUB_PRIMARY_ARTIFACT_MANIFEST",
    "/kaggle/input/datasets/pilkwang/biohub-tracking-support-pack-50ep-v1/ARTIFACT_MANIFEST.json"))
ALLOW_ARTIFACT_FALLBACK = os.environ.get("BIOHUB_ALLOW_ARTIFACT_FALLBACK", "0") != "0"


In [ ]:
import re

os.environ.setdefault("POLARS_PREFER_PKG", "32")

PACKAGE_SPECS = {
    "tracksdata": ("tracksdata", "tracksdata"),
    "zarr": ("zarr", "zarr>=3.0.10,<4"),
    "pyscipopt": ("pyscipopt", "pyscipopt"),
    "geff": ("geff", "geff>=1.1.3.1.1"),
    "geff_spec": ("geff_spec", "geff-spec<1.2"),
    "ilpy": ("ilpy", "ilpy>=0.5.1"),
    "polars": ("polars", "polars>=1.36"),
    "blosc2": ("blosc2", "blosc2"),
    "dask": ("dask", "dask"),
    "imagecodecs": ("imagecodecs", "imagecodecs"),
    "skimage": ("skimage", "scikit-image>=0.24"),
    "pyarrow": ("pyarrow", "pyarrow"),
    "rustworkx": ("rustworkx", "rustworkx>=0.17.1"),
    "sqlalchemy": ("sqlalchemy", "sqlalchemy>=2"),
    "numcodecs": ("numcodecs", "numcodecs>=0.13,<0.16"),
    "donfig": ("donfig", "donfig>=0.8"),
    "google_crc32c": ("google_crc32c", "google-crc32c>=1.5"),
    "bidict": ("bidict", "bidict>=0.23.1"),
    "psygnal": ("psygnal", "psygnal>=0.14"),
    "rich": ("rich", "rich"),
    "networkx": ("networkx", "networkx>=3.2.1"),
    "pydantic": ("pydantic", "pydantic>=2.11"),
    "pydantic_core": ("pydantic_core", "pydantic-core"),
    "annotated_types": ("annotated_types", "annotated-types"),
    "typing_extensions": ("typing_extensions", "typing-extensions>=4.13"),
    "typing_inspection": ("typing_inspection", "typing-inspection"),
    "markdown_it": ("markdown_it", "markdown-it-py"),
    "pygments": ("pygments", "pygments"),
    "click": ("click", "click"),
    "cloudpickle": ("cloudpickle", "cloudpickle"),
    "fsspec": ("fsspec", "fsspec"),
    "partd": ("partd", "partd"),
    "locket": ("locket", "locket"),
    "toolz": ("toolz", "toolz"),
    "yaml": ("yaml", "pyyaml"),
    "ndindex": ("ndindex", "ndindex"),
    "msgpack": ("msgpack", "msgpack"),
    "numexpr": ("numexpr", "numexpr"),
    "deprecated": ("deprecated", "deprecated"),
    "wrapt": ("wrapt", "wrapt"),
    "imageio": ("imageio", "imageio"),
    "PIL": ("PIL", "pillow"),
    "tifffile": ("tifffile", "tifffile"),
    "lazy_loader": ("lazy_loader", "lazy-loader"),
    "tqdm": ("tqdm", "tqdm"),
}
EXTRA_SPECS_BY_NAME = {
    "tracksdata": ["bidict>=0.23.1", "psygnal>=0.14", "rich"],
    "zarr": ["donfig>=0.8", "google-crc32c>=1.5", "numcodecs>=0.13,<0.16"],
    "geff": ["geff-spec<1.2", "networkx>=3.2.1", "pydantic>=2.11", "numcodecs>=0.13,<0.16"],
    "geff_spec": ["pydantic>=2.11", "annotated-types", "pydantic-core", "typing-inspection"],
    "polars": ["polars-runtime-32"],
    "dask": ["click", "cloudpickle", "fsspec", "partd", "pyyaml", "toolz"],
    "partd": ["locket"],
    "blosc2": ["ndindex", "msgpack", "numexpr"],
    "numcodecs": ["deprecated", "msgpack", "wrapt"],
    "rich": ["markdown-it-py", "pygments"],
    "pydantic": ["annotated-types", "pydantic-core", "typing-extensions>=4.13", "typing-inspection"],
    "skimage": ["imageio", "pillow", "tifffile", "lazy-loader", "networkx"],
}
PIP_DEPENDENCIES = [spec for _, spec in PACKAGE_SPECS.values()]
REQUIRED_MODULES = {name: module for name, (module, _) in PACKAGE_SPECS.items() if module}
FALLBACK_ARTIFACT_SLUGS = ["biohub-tracking-support-pack-v1"]
ALLOW_PIP_INSTALL = os.environ.get("BIOHUB_ALLOW_PIP_INSTALL", "0") != "0"


def module_missing(module_name: str) -> bool:
    return importlib.util.find_spec(module_name) is None


def has_model_artifact(path: Path) -> bool:
    has_repo_dir = (path / "repo").exists()
    has_weights_dir = (path / "weights" / METHOD / "split_0" / "edge_predictor_best.pth").exists()
    has_repo_zip = (path / "repo.zip").exists()
    has_weights_zip = (path / "weights.zip").exists()
    return (has_repo_dir and has_weights_dir) or (has_repo_zip and has_weights_zip)


def artifact_manifest(path: Path) -> dict:
    manifest = path / "ARTIFACT_MANIFEST.json"
    if not manifest.exists():
        return {}
    try:
        return json.loads(manifest.read_text())
    except Exception:
        return {}


def artifact_matches_target(path: Path) -> bool:
    if ALLOW_ARTIFACT_FALLBACK:
        return True
    manifest = artifact_manifest(path)
    artifact_name = str(manifest.get("artifact_name", ""))
    path_text = str(path)
    return TARGET_ARTIFACT_SLUG in {artifact_name, path.name} or TARGET_ARTIFACT_SLUG in path_text


def candidate_roots_for_slug(slug: str) -> list[Path]:
    return [
        Path(f"/kaggle/input/datasets/pilkwang/{slug}"),
        Path(f"/kaggle/input/{slug}"),
        Path(f"/kaggle/input/{slug}/{slug}"),
        Path(f"PublicNotebook/{slug}"),
    ]


def find_artifacts_root() -> Path:
    candidates: list[Path] = []
    for env_name in ["BIOHUB_MODEL_ARTIFACTS", "BIOHUB_ARTIFACTS"]:
        explicit = os.environ.get(env_name, "").strip()
        if explicit:
            candidates.append(Path(explicit))

    candidates.append(PRIMARY_ARTIFACT_MANIFEST.parent)
    candidates.extend(candidate_roots_for_slug(TARGET_ARTIFACT_SLUG))

    if ALLOW_ARTIFACT_FALLBACK:
        for slug in FALLBACK_ARTIFACT_SLUGS:
            candidates.extend(candidate_roots_for_slug(slug))

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if not child.is_dir():
                continue
            child_text = str(child)
            if TARGET_ARTIFACT_SLUG in child_text or ALLOW_ARTIFACT_FALLBACK:
                candidates.append(child)
                candidates.append(child / child.name)
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.append(grandchild)

    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if has_model_artifact(candidate) and artifact_matches_target(candidate):
            return candidate
    checked = "\n".join(str(path) for path in candidates[:80])
    raise FileNotFoundError(
        "Could not find the required model artifact. "
        f"Expected slug: {TARGET_ARTIFACT_SLUG}\n"
        "Attach the newly uploaded support dataset, or set BIOHUB_MODEL_ARTIFACTS.\n"
        "To debug with an older artifact, set BIOHUB_ALLOW_ARTIFACT_FALLBACK=1.\n"
        "Checked:\n" + checked
    )


def _has_package_file(path: Path) -> bool:
    if not path.exists() or not path.is_dir():
        return False
    patterns = ("*.whl", "*.tar.gz", "*.zip")
    return any(any(path.glob(pattern)) for pattern in patterns)


def find_offline_package_dirs(artifacts: Path) -> list[Path]:
    candidates: list[Path] = [
        artifacts / "wheels",
        artifacts,
        Path("/kaggle/working"),
        Path("/kaggle/working/wheels"),
    ]
    input_root = Path("/kaggle/input")
    if input_root.exists():
        for child in input_root.iterdir():
            if child.is_dir():
                candidates.extend([child / "wheels", child])
                for grandchild in child.iterdir():
                    if grandchild.is_dir():
                        candidates.extend([grandchild / "wheels", grandchild])

    out: list[Path] = []
    seen: set[Path] = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate in seen:
            continue
        seen.add(candidate)
        if _has_package_file(candidate):
            out.append(candidate)
    return out


def purge_imported_modules(package_names: list[str]) -> None:
    roots = {"tracksdata"}
    for name in package_names:
        if name in PACKAGE_SPECS:
            module = PACKAGE_SPECS[name][0]
            roots.add(module.split(".")[0])
        if name == "polars":
            roots.add("polars")
    for root in roots:
        for module_name in list(sys.modules):
            if module_name == root or module_name.startswith(root + "."):
                sys.modules.pop(module_name, None)


def polars_runtime_ready() -> bool:
    try:
        import polars as _pl
        from polars._plr import PySeries as _PySeries

        _ = _PySeries
        return hasattr(_pl, "Float16") and _pl.Series([-999999.0], dtype=_pl.Float64).dtype == _pl.Float64
    except Exception:
        return False


def packages_requiring_refresh() -> list[str]:
    refresh: list[str] = []
    if not module_missing("polars") and not polars_runtime_ready():
        refresh.append("polars")

    if not module_missing("zarr"):
        try:
            import zarr as _zarr
            version_text = str(getattr(_zarr, "__version__", "0"))
            major = int(version_text.split(".", 1)[0])
            if major < 3:
                refresh.append("zarr")
        except Exception:
            refresh.append("zarr")
    return refresh


def dependency_specs_for(missing: list[str]) -> list[str]:
    specs: list[str] = []
    seen: set[str] = set()

    def add(spec: str) -> None:
        key = spec.lower()
        if key not in seen:
            seen.add(key)
            specs.append(spec)

    for name in missing:
        if name in PACKAGE_SPECS:
            add(PACKAGE_SPECS[name][1])
        for spec in EXTRA_SPECS_BY_NAME.get(name, []):
            add(spec)
    return specs


def import_failures() -> dict[str, str]:
    failures: dict[str, str] = {}
    for name, module_name in REQUIRED_MODULES.items():
        try:
            importlib.import_module(module_name)
        except Exception as exc:
            failures[name] = f"{type(exc).__name__}: {exc}"
    return failures


def missing_names_from_failures(failures: dict[str, str]) -> list[str]:
    names: list[str] = []
    module_to_name = {module: name for name, module in REQUIRED_MODULES.items()}
    for message in failures.values():
        match = re.search(r"No module named ['\"]([^'\"]+)['\"]", message)
        if match:
            module = match.group(1).split(".")[0]
        else:
            match = re.search(r"module ['\"]([^'\"]+)['\"] has no attribute", message)
            if not match:
                continue
            module = match.group(1).split(".")[0]
        name = module_to_name.get(module)
        if name and name not in names:
            names.append(name)
    return names


def install_missing_dependencies(missing: list[str], artifacts: Path) -> None:
    specs = dependency_specs_for(missing)
    force_reinstall = bool({"polars", "zarr"} & set(missing))
    if not specs:
        return

    package_dirs = find_offline_package_dirs(artifacts)
    if package_dirs:
        offline_cmd = [sys.executable, "-m", "pip", "install", "--no-index", "--no-deps"]
        if force_reinstall:
            offline_cmd.append("--force-reinstall")
        for package_dir in package_dirs:
            offline_cmd.extend(["--find-links", str(package_dir)])
        offline_cmd.extend(specs)
        print("Installing missing packages from offline package dirs:", missing)
        print("Dependency resolver is disabled with --no-deps to avoid replacing Kaggle numpy/scipy in a live kernel.")
        print("Offline package dirs:", [str(path) for path in package_dirs])
        result = subprocess.run(offline_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("Offline dependency install succeeded.")
            return
        print("Offline dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    if ALLOW_PIP_INSTALL:
        online_cmd = [sys.executable, "-m", "pip", "install", "--no-deps"]
        if force_reinstall:
            online_cmd.append("--force-reinstall")
        online_cmd.extend(specs)
        print("Installing missing packages from PyPI:", missing)
        result = subprocess.run(online_cmd, text=True, capture_output=True)
        if result.returncode == 0:
            purge_imported_modules(missing)
            print("PyPI dependency install succeeded.")
            return
        print("PyPI dependency install failed. Last pip output:")
        print((result.stdout or "")[-2000:])
        print((result.stderr or "")[-2000:])

    command = "pip install tracksdata zarr>=3.0.10,<4 pyscipopt geff geff-spec ilpy polars blosc2 dask imagecodecs pyarrow rustworkx sqlalchemy donfig numcodecs"
    raise ImportError(
        "Missing required packages or dependency wheels: " + ", ".join(missing) + "\n"
        "Attach the support dataset with offline wheels. If supplying Kaggle dependency input instead, use:\n"
        + command + "\n"
        "Do not quote zarr>=3.0.10,<4 in Kaggle dependency input."
    )


def ensure_dependencies(artifacts: Path) -> None:
    for _ in range(5):
        refresh = packages_requiring_refresh()
        if refresh:
            install_missing_dependencies(refresh, artifacts)
            continue

        missing = [pkg for pkg, module in REQUIRED_MODULES.items() if module_missing(module)]
        if missing:
            install_missing_dependencies(missing, artifacts)
            continue

        failures = import_failures()
        if not failures:
            print("Required graph/Zarr/ILP packages import successfully.")
            return

        missing_from_import = missing_names_from_failures(failures)
        if missing_from_import:
            install_missing_dependencies(missing_from_import, artifacts)
            continue

        raise ImportError(
            "Required packages are present but failed to import. "
            "This may indicate a binary dependency mismatch in the live notebook kernel. "
            "Keep Kaggle dependency input empty and attach the wheels artifact.\n"
            + json.dumps(failures, indent=2)
        )

    failures = import_failures()
    raise ImportError(
        "Dependency recovery did not converge after repeated offline installs. "
        "The attached support artifact may be missing wheels.\n"
        + json.dumps(failures, indent=2)
    )


def remove_path(path: Path) -> None:
    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        shutil.rmtree(path)


def copy_or_extract_tree(src_dir: Path, src_zip: Path, dst: Path) -> None:
    remove_path(dst)
    if src_dir.exists() and src_dir.is_dir():
        shutil.copytree(src_dir, dst)
        return
    if src_zip.exists() and src_zip.is_file():
        dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(src_zip) as zf:
            zf.extractall(dst)
        return
    raise FileNotFoundError(f"Missing source tree or zip: {src_dir} / {src_zip}")


def link_or_copy_tree(src: Path, dst: Path) -> None:
    remove_path(dst)
    try:
        os.symlink(src, dst, target_is_directory=True)
    except Exception:
        shutil.copytree(src, dst)


def materialize_inference_repo(artifacts: Path) -> None:
    copy_or_extract_tree(artifacts / "repo", artifacts / "repo.zip", REPO_DIR)

    weights_src = artifacts / "weights"
    weights_zip = artifacts / "weights.zip"
    weights_dst = REPO_DIR / "weights"
    if weights_src.exists() and weights_src.is_dir():
        link_or_copy_tree(weights_src, weights_dst)
    elif weights_zip.exists() and weights_zip.is_file():
        remove_path(weights_dst)
        weights_dst.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(weights_zip) as zf:
            zf.extractall(weights_dst)
    else:
        raise FileNotFoundError(f"Missing weights tree or zip under {artifacts}")

    required = [
        REPO_DIR / "scripts" / "predict_unet_transformer.py",
        REPO_DIR / WEIGHTS_RELATIVE,
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Materialized inference repo is incomplete:\n" + "\n".join(missing))
    print("Inference repo:", REPO_DIR)
    print("Weights:", REPO_DIR / WEIGHTS_RELATIVE)


ARTIFACTS = find_artifacts_root()
print("ARTIFACTS:", ARTIFACTS)
print("Has offline wheels:", (ARTIFACTS / "wheels").exists())
manifest_info = artifact_manifest(ARTIFACTS)
if manifest_info:
    print("Artifact name:", manifest_info.get("artifact_name"))
    print("Weight sha256:", manifest_info.get("model", {}).get("weight_sha256"))
    print("Weight path:", manifest_info.get("model", {}).get("weight_path"))
    _expected_primary_sha256 = "12f6881ee3620a831697ca098ff8f48e687a24225f4e048b538deec3562fe771"
    _actual_primary_sha256 = str(manifest_info.get("model", {}).get("weight_sha256", ""))
    if _actual_primary_sha256 != _expected_primary_sha256:
        raise RuntimeError(
            "Primary model checksum mismatch: "
            f"expected {_expected_primary_sha256}, got {_actual_primary_sha256 or 'missing'}"
        )

ensure_dependencies(ARTIFACTS)
materialize_inference_repo(ARTIFACTS)


In [ ]:
"""Lightweight graph post-processing for prediction .geff files (CPU, local).

Ops (all optional, applied in this order):
  1. gap closing      : link track end at t to track start at t+2 (insert a midpoint node) if within gap_um
  2. prune isolated   : drop nodes with no edges
  3. short components : drop weakly-connected components with fewer than min_len nodes (keep ones with a division if keep_div)

Usage:
  python tools/postproc.py IN_DIR OUT_DIR --min-len 6 --gap-um 5.0
  from tools.postproc import process_dir
"""
from __future__ import annotations
import argparse, math, shutil
from pathlib import Path
import numpy as np
import networkx as nx
import geff
from scipy.spatial import cKDTree

VOXEL = np.array([1.625, 0.40625, 0.40625])  # z, y, x um


def load(path: Path) -> nx.DiGraph:
    g, _meta = geff.read(path, structure_validation=False)
    assert isinstance(g, nx.DiGraph)
    # keep only ILP-solution edges/nodes when the flag is present
    if any("solution" in d for _, d in g.nodes(data=True)):
        g.remove_nodes_from([n for n, d in g.nodes(data=True) if d.get("solution") is False])
    if any("solution" in d for _, _, d in g.edges(data=True)):
        g.remove_edges_from([(u, v) for u, v, d in g.edges(data=True) if d.get("solution") is False])
    return g


def save(g: nx.DiGraph, path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    for n, d in g.nodes(data=True):
        for k in ("t", "z", "y", "x"):
            d[k] = float(d[k]) if k != "t" else int(d["t"])
        d.pop("solution", None)
    for u, v, d in g.edges(data=True):
        d.pop("solution", None)
    geff.write(g, path, axis_names=["t", "z", "y", "x"], axis_types=["time", "space", "space", "space"],
               zarr_format=3, structure_validation=False)


def _pos(d) -> np.ndarray:
    return np.array([d["z"], d["y"], d["x"]], dtype=float) * VOXEL


def _velocity(g: nx.DiGraph, n, k: int = 3) -> np.ndarray:
    """EMA-like mean displacement (um/frame) over the last k edges ending at n; zeros if none."""
    disp = []
    cur = n
    for _ in range(k):
        preds = list(g.predecessors(cur))
        if not preds:
            break
        prev = preds[0]
        disp.append(_pos(g.nodes[cur]) - _pos(g.nodes[prev]))
        cur = prev
    if not disp:
        return np.zeros(3)
    w = np.array([0.5 ** i for i in range(len(disp))]); w /= w.sum()
    return (np.stack(disp) * w[:, None]).sum(axis=0)


def gap_close(g: nx.DiGraph, gap_um: float, max_gap: int = 1, min_prob: float | None = None,
              velocity: float = 0.0, direct_um: float = 0.0) -> int:
    """direct_um > 0 additionally links track ends at t to track starts at t+1 (no gap) within direct_um (mutual NN)."""
    """Connect track ends (out-degree 0) at t to track starts (in-degree 0) at t+gap+1 by nearest neighbour
    within gap_um (um, per frame step scaled by gap+1), inserting synthetic midpoint nodes. Greedy, mutual-NN."""
    added = 0
    next_id = max(g.nodes) + 1 if g.number_of_nodes() else 0
    ends: dict[int, list] = {}
    starts: dict[int, list] = {}
    for n, d in g.nodes(data=True):
        if g.out_degree(n) == 0:
            ends.setdefault(int(d["t"]), []).append(n)
        if g.in_degree(n) == 0:
            starts.setdefault(int(d["t"]), []).append(n)
    for gap in range(0 if direct_um > 0 else 1, max_gap + 1):
        for t, e_nodes in sorted(ends.items()):
            s_nodes = starts.get(t + gap + 1, [])
            e_nodes = [n for n in e_nodes if g.out_degree(n) == 0]
            s_nodes = [n for n in s_nodes if g.in_degree(n) == 0]
            if not e_nodes or not s_nodes:
                continue
            ep = np.stack([_pos(g.nodes[n]) for n in e_nodes]); sp = np.stack([_pos(g.nodes[n]) for n in s_nodes])
            if velocity > 0:  # project track ends forward by their recent velocity
                ep = ep + velocity * (gap + 1) * np.stack([_velocity(g, n) for n in e_nodes])
            te = cKDTree(ep); ts = cKDTree(sp)
            r = direct_um if gap == 0 else gap_um * (gap + 1)
            d_es, i_es = ts.query(ep, k=1, distance_upper_bound=r)
            d_se, i_se = te.query(sp, k=1, distance_upper_bound=r)
            for ei, (dist, si) in enumerate(zip(d_es, i_es)):
                if not np.isfinite(dist) or si >= len(s_nodes):
                    continue
                if i_se[si] != ei:  # mutual nearest neighbour
                    continue
                a, b = e_nodes[ei], s_nodes[si]
                da, db = g.nodes[a], g.nodes[b]
                prev = a
                for k in range(1, gap + 1):
                    f = k / (gap + 1)
                    nid = next_id; next_id += 1
                    g.add_node(nid, t=int(da["t"]) + k,
                               z=float(da["z"] + f * (db["z"] - da["z"])),
                               y=float(da["y"] + f * (db["y"] - da["y"])),
                               x=float(da["x"] + f * (db["x"] - da["x"])))
                    g.add_edge(prev, nid, edge_prob=0.5); prev = nid
                g.add_edge(prev, b, edge_prob=0.5)
                added += 1
    return added


def add_divisions(g: nx.DiGraph, parent_um: float = 7.0, sister_um: float = 12.0, max_frac: float = 0.01,
                  steal_prob: float = 0.0) -> int:
    """Post-hoc divisions: a node p at t with exactly one child c (t+1) gets a second child q if q is a
    track START at t+1 (in-degree 0), q is p's nearest unclaimed start within parent_um, the sisters c,q are within
    sister_um, and p is q's nearest single-child parent. Caps additions to max_frac of nodes. Returns #added."""
    starts: dict[int, list] = {}
    parents: dict[int, list] = {}
    for n, d in g.nodes(data=True):
        t = int(d["t"])
        if g.in_degree(n) == 0 and g.out_degree(n) >= 1:
            starts.setdefault(t, []).append(n)
        elif steal_prob > 0 and g.in_degree(n) == 1 and g.out_degree(n) >= 1:
            # weakly linked node: eligible to be re-assigned as a second daughter
            pr = next(iter(g.predecessors(n)))
            if float(g.edges[pr, n].get("edge_prob", 1.0)) < steal_prob:
                starts.setdefault(t, []).append(n)
        if g.out_degree(n) == 1:
            parents.setdefault(t, []).append(n)
    cap = max(1, int(max_frac * g.number_of_nodes()))
    added = 0
    for t, p_nodes in sorted(parents.items()):
        s_nodes = starts.get(t + 1, [])
        if not s_nodes:
            continue
        pp = np.stack([_pos(g.nodes[n]) for n in p_nodes]); sp = np.stack([_pos(g.nodes[n]) for n in s_nodes])
        tp, ts = cKDTree(pp), cKDTree(sp)
        d_ps, i_ps = ts.query(pp, k=1, distance_upper_bound=parent_um)
        d_sp, i_sp = tp.query(sp, k=1, distance_upper_bound=parent_um)
        for pi, (dist, si) in enumerate(zip(d_ps, i_ps)):
            if not np.isfinite(dist) or si >= len(s_nodes) or i_sp[si] != pi:
                continue
            p, q = p_nodes[pi], s_nodes[si]
            if g.out_degree(p) != 1 or g.in_degree(q) > 1:
                continue
            c = next(iter(g.successors(p)))
            if c == q or np.linalg.norm(_pos(g.nodes[c]) - _pos(g.nodes[q])) > sister_um:
                continue
            if g.in_degree(q) == 1:  # steal: q currently has a weak parent
                old_p = next(iter(g.predecessors(q)))
                if old_p == p or g.out_degree(old_p) > 1:
                    continue
                g.remove_edge(old_p, q)
            g.add_edge(p, q, edge_prob=0.5); added += 1
            if added >= cap:
                return added
    return added


def prune_isolated(g: nx.DiGraph) -> int:
    iso = [n for n in g.nodes if g.degree(n) == 0]
    g.remove_nodes_from(iso)
    return len(iso)


def filter_short(g: nx.DiGraph, min_len: int, keep_div: bool = True) -> int:
    removed = 0
    for comp in list(nx.weakly_connected_components(g)):
        if len(comp) >= min_len:
            continue
        if keep_div and any(g.out_degree(n) >= 2 for n in comp):
            continue
        g.remove_nodes_from(comp); removed += 1
    return removed


def process(g: nx.DiGraph, min_len: int = 0, gap_um: float = 0.0, max_gap: int = 1, keep_div: bool = True,
            div_parent_um: float = 0.0, div_sister_um: float = 12.0, div_max_frac: float = 0.01,
            gap_velocity: float = 0.0, div_steal_prob: float = 0.0, direct_um: float = 0.0) -> dict:
    stats = {}
    if gap_um > 0 or direct_um > 0:
        stats["gaps_closed"] = gap_close(g, max(gap_um, 1e-6), max_gap if gap_um > 0 else 0, velocity=gap_velocity, direct_um=direct_um)
    if div_parent_um > 0:
        stats["divisions_added"] = add_divisions(g, div_parent_um, div_sister_um, div_max_frac, steal_prob=div_steal_prob)
    stats["isolated"] = prune_isolated(g)
    if min_len > 1:
        stats["short_removed"] = filter_short(g, min_len, keep_div)
    stats["nodes"] = g.number_of_nodes(); stats["edges"] = g.number_of_edges()
    return stats


def process_dir(in_dir: Path, out_dir: Path, **kw) -> list[Path]:
    out_dir.mkdir(parents=True, exist_ok=True)
    outs = []
    for p in sorted(Path(in_dir).glob("*.geff")):
        g = load(p)
        st = process(g, **kw)
        o = out_dir / p.name
        save(g, o); outs.append(o)
        print(f"  {p.stem}: {st}", flush=True)
    return outs




In [ ]:
# --- exp10: clean pipeline submission: fine-tuned weights -> predict (ILP) -> postproc -> submission.csv ---
import csv, json, os, subprocess, sys, time, shutil
from pathlib import Path
E = os.environ.get
WEIGHTS_SUB = E("BIOHUB_WEIGHTS_SUB", "exp03B")
DET_THRESHOLD = E("BIOHUB_DET_THRESHOLD", "0.965")
ILP_ARGS = ["--use-ilp", "--ilp-edge-weight", "-1.0", "--ilp-appearance-weight", E("BIOHUB_ILP_APPEARANCE_WEIGHT", "0.0"),
            "--ilp-disappearance-weight", E("BIOHUB_ILP_DISAPPEARANCE_WEIGHT", "2"),
            "--ilp-division-weight", E("BIOHUB_ILP_DIVISION_WEIGHT", "1.2")]
PP = dict(gap_um=float(E("BIOHUB_PP_GAP_UM", "5.0")), max_gap=int(E("BIOHUB_PP_MAX_GAP", "1")),
          div_parent_um=float(E("BIOHUB_PP_DIV_PARENT_UM", "5.0")), div_sister_um=float(E("BIOHUB_PP_DIV_SISTER_UM", "12.0")),
          div_max_frac=float(E("BIOHUB_PP_DIV_MAX_FRAC", "0.01")), min_len=int(E("BIOHUB_PP_MIN_LEN", "0")),
          direct_um=float(E("BIOHUB_PP_DIRECT_UM", "8.0")))
VALIDATE = E("BIOHUB_SUBMIT_VALIDATE", "0") != "0"
HOLDOUT = json.loads(E("BIOHUB_HOLDOUT_JSON", "{}")).get("holdout", [])
N_GPU = int(E("BIOHUB_N_GPU", "2"))
SUBMISSION_PATH = WORKING_DIR / "submission.csv"

# ---- writable repo + weights ----
TRAIN_REPO = WORKING_DIR / "train_repo"
shutil.rmtree(TRAIN_REPO, ignore_errors=True); TRAIN_REPO.mkdir(parents=True)
for sub in ("scripts", "src"):
    shutil.copytree(REPO_DIR / sub, TRAIN_REPO / sub)
wsrc = next(p for p in [Path("/kaggle/input/datasets/vitaliineibauer/biohub-ft-weights") / WEIGHTS_SUB,
                        Path("/kaggle/input/biohub-ft-weights") / WEIGHTS_SUB] if p.exists())
WDIR = TRAIN_REPO / "weights" / "ft" / "split_0"; WDIR.mkdir(parents=True)
for f in ("edge_predictor_best.pth", "config.json"):
    shutil.copy(wsrc / f, WDIR / f)
WEIGHTS = WDIR / "edge_predictor_best.pth"
print("weights:", wsrc, (WDIR / "config.json").read_text(), flush=True)
ENV = {**os.environ, "PYTHONPATH": "src", "PYTHONUNBUFFERED": "1"}
SCORE_SCRIPT = '''
import argparse, json, sys
from pathlib import Path
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
import evaluate as ev
ap = argparse.ArgumentParser(); ap.add_argument("--method"); ap.add_argument("--out"); ap.add_argument("geffs", nargs="+")
a = ap.parse_args()
rows = ev.evaluate_run({"username": "x", "method": a.method, "split": "split_0", "dir": None, "geffs": [Path(g) for g in a.geffs]})
s = ev.summarise(rows)
summary = {k: (float(v) if isinstance(v, (int, float)) else v) for k, v in s.items()}
Path(a.out).write_text(json.dumps({"summary": summary, "rows": rows}, indent=1, default=str))
print("SCORE", a.method, json.dumps(summary))
'''
(TRAIN_REPO / "scripts" / "score_holdout.py").write_text(SCORE_SCRIPT)


def predict(data_dir: Path, stems: list, method: str) -> list:
    splits_path = TRAIN_REPO / f"splits_{method}.json"
    splits_path.write_text(json.dumps([{"split": 0, "train": [], "test": stems}]))
    procs = []
    for g in range(N_GPU):
        cmd = [sys.executable, "scripts/predict_unet_transformer.py", "--data-dir", str(data_dir), "--splits", str(splits_path),
               "--split", "0", "--weights", str(WEIGHTS), "--method", f"{method}_gpu{g}", "--slice", f"{g}::{N_GPU}",
               "--det-threshold", DET_THRESHOLD, "--unet-batch-size", "4", *ILP_ARGS]
        procs.append(subprocess.Popen(cmd, cwd=TRAIN_REPO, env={**ENV, "CUDA_VISIBLE_DEVICES": str(g), "BIOHUB_DATA_DIR": str(data_dir)},
                                      stdout=open(WORKING_DIR / f"predict_{method}_gpu{g}.log", "w"), stderr=subprocess.STDOUT))
    t0 = time.time()
    for g, p in enumerate(procs):
        if p.wait() != 0:
            print((WORKING_DIR / f"predict_{method}_gpu{g}.log").read_text()[-3000:])
            raise RuntimeError(f"predict shard {g} failed for {method}")
    geffs = sorted((TRAIN_REPO / "predictions").rglob(f"{method}_gpu*/split_0/*.geff"))
    print(f"{method}: {len(geffs)} graphs in {(time.time()-t0)/60:.1f} min", flush=True)
    assert {g.stem for g in geffs} == set(stems), "missing predictions"
    return geffs


def postprocess(geffs: list, out_dir: Path) -> list:
    out_dir.mkdir(parents=True, exist_ok=True)
    outs = []
    t0 = time.time()
    for p in geffs:
        g = load(p)
        st = process(g, **PP)
        o = out_dir / p.name
        save(g, o); outs.append(o)
        print(f"  pp {p.stem}: {st}", flush=True)
    print(f"post-processing {len(outs)} graphs in {(time.time()-t0)/60:.1f} min", flush=True)
    return outs


# ---- optional self-check on holdout (train videos) ----
if VALIDATE and HOLDOUT:
    raw = predict(TRAIN_DIR, HOLDOUT, "val_raw")
    pp = postprocess(raw, WORKING_DIR / "val_pp")
    for name, gl in (("val_raw", raw), ("val_pp", pp)):
        out = WORKING_DIR / f"score_{name}.json"
        r = subprocess.run([sys.executable, "scripts/score_holdout.py", "--method", name, "--out", str(out), *map(str, gl)],
                           cwd=TRAIN_REPO, env={**ENV, "BIOHUB_DATA_DIR": str(TRAIN_DIR)}, capture_output=True, text=True)
        if r.returncode:
            print(r.stdout[-1500:], r.stderr[-1500:])
        s = json.loads(out.read_text())["summary"]
        print(f"== {name}: score={s['score']:.4f} edge={s['edge_jaccard']:.4f} div={int(s['division_tp'])}/{int(s['division_fp'])}/{int(s['division_fn'])}", flush=True)
    shutil.rmtree(WORKING_DIR / "val_pp", ignore_errors=True)

# ---- test prediction + submission ----
test_stems = sorted(p.name[:-5] for p in TEST_DIR.iterdir() if p.name.endswith(".zarr"))
print("test videos:", len(test_stems), flush=True)
raw = predict(TEST_DIR, test_stems, "test")
pp = postprocess(raw, WORKING_DIR / "test_pp")

COLUMNS = ["id", "dataset", "row_type", "node_id", "t", "z", "y", "x", "source_id", "target_id"]
row_id = 0; n_nodes = 0; n_edges = 0
with open(SUBMISSION_PATH, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLUMNS); w.writeheader()
    for p in pp:
        g = load(p)
        ds = p.stem
        for n, d in sorted(g.nodes(data=True), key=lambda kv: (int(kv[1]["t"]), int(kv[0]))):
            w.writerow({"id": row_id, "dataset": ds, "row_type": "node", "node_id": int(n), "t": int(d["t"]),
                        "z": max(0, int(round(float(d["z"])))), "y": max(0, int(round(float(d["y"])))), "x": max(0, int(round(float(d["x"])))),
                        "source_id": -1, "target_id": -1}); row_id += 1; n_nodes += 1
        for u, v in g.edges():
            assert int(g.nodes[v]["t"]) == int(g.nodes[u]["t"]) + 1
            w.writerow({"id": row_id, "dataset": ds, "row_type": "edge", "node_id": -1, "t": -1, "z": -1, "y": -1, "x": -1,
                        "source_id": int(u), "target_id": int(v)}); row_id += 1; n_edges += 1
print(f"wrote {SUBMISSION_PATH}: {n_nodes} nodes, {n_edges} edges, {row_id} rows", flush=True)
import pandas as pd
df = pd.read_csv(SUBMISSION_PATH)
assert list(df.columns) == COLUMNS and df["id"].tolist() == list(range(len(df)))
assert set(df["dataset"].unique()) == set(test_stems)
print(df.head(6).to_string())
for d in (TRAIN_REPO, REPO_DIR, WORKING_DIR / "test_pp"):
    shutil.rmtree(d, ignore_errors=True)
print("done; kept:", sorted(p.name for p in WORKING_DIR.iterdir()))
